# LLM 최적화의 핵심 인과관계

LLM 최적화의 역사를 관통하는 **핵심 인과관계**를 3단계로 정리하면 다음과 같습니다.

---

## 1단계: 속도를 위해 메모리를 희생하다 (KV Cache의 등장)

LLM은 기본적으로 **"이전 대화 내용을 다 기억해야"** 다음 단어를 뱉을 수 있습니다.

- **문제:** 단어를 하나 생성할 때마다 처음부터 끝까지 다시 계산 → 너무 느림 (연산 낭비)  
- **해결 (KV Cache):** 이전에 계산한 값(Key, Value)을 메모리에 저장 → 신규 단어만 계산  
- **결과:** 추론 속도가 비약적으로 빨라짐  
- **부작용:** **메모리(VRAM) 사용량 폭발** → 대화가 길어질수록 캐시 크기 증가  

---

## 2단계: 메모리 관리의 비효율 (기존 문제)

vLLM 이전에는 KV Cache 저장 방식이 매우 비효율적이었습니다. (운영체제로 치면 `malloc`을 무식하게 사용한 셈)

- **연속 할당의 저주:** "사용자가 2048토큰까지 생성할 수 있으니, 미리 **2048칸짜리 연속 메모리**를 예약"  
- **참사:** 사용자가 "안녕" 하고 끝내면? 예약된 2046칸은 그대로 버려짐 (**내부 단편화**)  
- **결과:** KV Cache 덕에 속도는 빨라졌지만, **메모리 낭비가 60~80%** → GPU에 Batch를 많이 못 태움  

---

## 3단계: PagedAttention으로 해결 (vLLM의 혁신)

여기서 vLLM이 등장합니다.

- **해결 (PagedAttention):** "연속된 자리 예약 대신, **OS 페이징처럼 쪼개서 빈 공간에 할당**"  
- **효과:**  
  1. 미리 자리를 맡아둘 필요 없음 (On-demand 할당)  
  2. 메모리 낭비가 거의 '0'에 수렴  
  3. **남는 메모리에 더 많은 사용자(Request)를 동시에 처리 가능**  

---

## 🎯 핵심 요약: 속도 vs 처리량

**"빨라졌다"**의 의미를 두 가지로 구분해야 합니다.

1. **KV Cache:** **한 명의 답변 속도(Latency)**를 빠르게 함 (재계산 제거)  
2. **PagedAttention:** **동시 처리량(Throughput)**을 늘림 (메모리 효율 개선)  

---

## ✅ 결론

> **"KV Cache로 (연산을 줄여) 빨라졌지만 메모리가 터져나갔고, 그 메모리 효율 문제를 PagedAttention(페이징 기법)으로 해결했다."**

이것이 vLLM이 현재 추론 프레임워크의 왕좌를 차지한 정확한 이유입니다.

# vLLM 실습: Colab

1. vLLM의 핵심 개념과 Ollama와의 차이점 이해
2. Offline Inference로 배치 추론 실행
3. OpenAI 호환 API 서버 구동 및 테스트
4. 다양한 파라미터 실험을 통한 성능 체감

---

**런타임 유형 변경 → GPU (A100)**



In [ ]:
!nvidia-smi

Mon Jan 26 10:40:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU Memory: 39.6 GB


In [ ]:
from huggingface_hub import login
import getpass

hf_token = getpass.getpass("Hugging Face Token 입력: ")
login(token=hf_token)

Hugging Face Token 입력: ··········


### 1.2 vLLM 설치

- 설치에 5분이상

In [ ]:
# vLLM
!pip -q uninstall -y tensorflow-decision-forests google-generativeai google-ai-generativelanguage tensorflow grpcio-status
!pip -q install -U "protobuf>=6.33.0,<7" vllm==0.14.0

# OpenAI
!pip install openai -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 355.0/355.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 134.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 117.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 115.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8

In [ ]:
import vllm
print(f"vLLM version: {vllm.__version__}")

vLLM version: 0.14.0


### 💡 vLLM vs Ollama 비교

| 특성 | Ollama | vLLM |
|------|--------|------|
| 목적 | 로컬 개발/테스트 | 프로덕션 서빙 |
| 설치 | 원클릭 | pip install |
| 최적화 | 기본 | PagedAttention, Continuous Batching |
| 동시 요청 | 순차 처리 | 효율적 배치 처리 |
| 모델 포맷 | GGUF | HuggingFace, AWQ, GPTQ |

---

## Part 2: Offline Inference

vLLM의 가장 기본적인 사용법으로, API 서버 없이 직접 추론을 실행

### 2.1 모델 로드

In [ ]:
# !pip -q uninstall -y tensorflow-decision-forests google-generativeai google-ai-generativelanguage tensorflow grpcio-status
# !pip -q install -U "protobuf>=6.33.0,<7" vllm==0.14.0

In [ ]:
import os  # 운영체제(OS)의 기능을 사용하기 위한 모듈 임포트 (환경변수 설정 등)
import multiprocessing as mp  # 멀티프로세싱(병렬 처리)을 위한 모듈 임포트

# =================================================================
# [이 셀의 역할: AI 모델 로딩 및 엔진 초기화]
# 1. 안전 장치: GPU 연산 시 프로세스 충돌을 방지하기 위한 설정 (Spawn)
# 2. 엔진 설정: 모델 선택, 메모리 할당량, 최대 문장 길이 등을 결정
# 3. 실질적 로드: 모델 파일을 다운로드하여 GPU 메모리(VRAM)에 올리는 "로딩 화면" 단계
# =================================================================

# [중요] Colab/Jupyter 환경 설정
# vLLM은 성능을 위해 여러 프로세스를 생성하는데, Colab에서는 'fork' 방식이 문제를 일으킬 수 있어 'spawn'으로 강제합니다.
# - Fork: 부모 프로세스를 그대로 복사 (빠르지만 GPU 자원이 꼬여서 먹통이 될 위험이 큼)
# - Spawn: 새로운 파이썬 인터프리터를 깔끔하게 새로 시작 (안전하며 vLLM에서 가장 권장하는 방식)
# 이 설정은 반드시 vLLM을 임포트하기 전에 수행되어야 적용됩니다.

# 1. 환경 변수 설정: vLLM 내부 작업자가 'spawn' 방식을 쓰도록 지시 (안전한 병렬 처리)
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# 2. 파이썬 멀티프로세싱 설정: 프로세스 시작 방식을 'spawn'으로 강제하고, 이미 설정되어 있어도 덮어씀
mp.set_start_method("spawn", force=True)

# vLLM 라이브러리에서 핵심 클래스 임포트
# LLM: 거대언어모델을 로드하고 추론을 담당하는 메인 엔진 클래스
# SamplingParams: 텍스트 생성 시 확률적 샘플링(온도, Top-p 등)을 제어하는 설정 클래스
from vllm import LLM, SamplingParams

# [모델 로드 및 엔진 초기화]
# LLM 클래스의 인스턴스(객체)를 생성합니다. 이 과정에서 모델 가중치(Weight)를 GPU로 다운로드 및 로드합니다.
llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",  # HuggingFace Hub에 있는 모델 ID (또는 로컬 경로)
    dtype="half",                        # 데이터 타입 설정. 'half'(FP16)를 사용하여 성능과 메모리 효율을 모두 잡음
    gpu_memory_utilization=0.75,         # GPU 메모리의 75%를 vLLM 전용으로 예약 (나머지 25%는 시스템 및 기타 연산용)
    max_model_len=4096,                  # 모델이 한 번에 처리할 수 있는 최대 토큰(단어 조각) 길이
)

# 객체 생성이 완료되면(데이터가 GPU에 모두 올라가면) 아래 메시지가 출력됩니다.
print("--- 모델 로드 완료! 이제 질문을 던질 준비가 되었습니다.")

INFO 01-26 10:44:08 [utils.py:263] non-default args: {'dtype': 'half', 'max_model_len': 4096, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

INFO 01-26 10:44:22 [model.py:530] Resolved architecture: Qwen2ForCausalLM
WARNING 01-26 10:44:22 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-26 10:44:22 [model.py:1545] Using max model len 4096
INFO 01-26 10:44:22 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 01-26 10:44:22 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-26 10:44:22 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

INFO 01-26 10:45:50 [llm.py:347] Supported tasks: ['generate']
--- 모델 로드 완료!


### 2.2 단일 프롬프트 추론

In [ ]:
# [1. 샘플링 파라미터 설정]
# SamplingParams 객체를 생성하여 생성 전략을 정의합니다.
# 이 객체는 나중에 llm.generate() 함수에 인자로 전달됩니다.
sampling_params = SamplingParams(
    temperature=0.7,   # 0.0 ~ 1.0 사이 값. 높을수록 창의적(랜덤성 증가), 낮을수록 결정적(논리적)
    top_p=0.9,         # 누적 확률 90% 내의 단어들 중에서만 선택 (Nucleus Sampling)
    max_tokens=256     # 생성할 최대 토큰(단어 조각) 수 제한
)

# [2. 입력 데이터 준비]
# 모델에게 입력할 질문(Prompt)을 문자열 변수에 저장합니다.
prompt = "인공지능이 교육 분야에 미치는 영향에 대해 설명해주세요."

# [3. 추론 실행 (Generation)]
# llm.generate() 함수는 리스트 형태의 프롬프트를 받습니다. -> [prompt]
# 반환값은 RequestOutput 객체들의 리스트입니다.
# 여기서는 프롬프트가 1개이므로 결과 리스트의 첫 번째 요소([0])를 가져와 output 변수에 저장합니다.
output = llm.generate([prompt], sampling_params)[0]

# [4. 결과 출력]
# output 객체 내부 구조:
# - output.prompt: 입력했던 프롬프트
# - output.outputs: 생성된 출력 후보들의 리스트 (n=1일 경우 1개)
# - output.outputs[0].text: 실제 생성된 텍스트 문자열

print("--- 프롬프트:")
print(prompt)  # 입력 출력
print("\n--- 응답:")
print(output.outputs[0].text)  # 모델이 생성한 텍스트 출력

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

--- 프롬프트:
인공지능이 교육 분야에 미치는 영향에 대해 설명해주세요.

--- 응답:
 인공지능(AI)이 교육 분야에 미치는 영향은 여러 가지로 나타나고 있습니다. 이에 대해 자세히 살펴보도록 하겠습니다.

1. 교육 기반의 변화: 인공지능은 다양한 교육 기술을 개선하고 기능을 강화하는 데 큰 도움이 됩니다. 예를 들어, AI가 지도 학습을 돕고, 학습 과정을 자동화하고, 학습 모델을 개선하는 데 기여할 수 있습니다.

2. 개인화된 교육: AI는 학생의 특성과 필요에 따라 맞춤형 교육을 제공할 수 있습니다. 예를 들어, 학생의 학습 패턴을 분석하여 개인화된 학습 콘텐츠를 제공할 수 있습니다.

3. 풍부한 교육 자료: AI는 학습 자료를 생성하고 유지하는 데 도움이 됩니다. 예를 들어, 학생들이 새로운 개념을 이해하는 데 필요한 콘텐츠를 생성할 수 있습니다.

4. 학습에 대한


### 2.3 배치 추론 (vLLM의 핵심!!!!)

여러 프롬프트를 동시에 처리하여 처리량(throughput)을 극대화

In [ ]:
import time  # 시간 측정을 위한 모듈

# =================================================================
# [이 셀의 핵심: 배치 처리(Batch Processing)와 병렬 추론]
# 1. 병렬 처리 (Parallelism): GPU는 수천 개의 코어를 가진 '거대한 그릴'과 같습니다.
#    - 하나씩 차례대로 굽는 게 아니라, 5개의 스테이크(질문)를 동시에 올려 한꺼번에 뒤집습니다.
# 2. 행렬 연산 (Matrix): 5개의 문장을 하나의 큰 행렬로 묶어 계산하므로 효율이 극대화됩니다.
# 3. Continuous Batching: vLLM의 핵심 기술로, 먼저 끝난 답변은 바로 내보내고
#    그 빈자리에 새 질문을 즉시 채워 넣어 GPU가 쉴 틈 없이 일하게 만듭니다.
# 4. 효율성: 5개를 동시에 처리해도 1개를 처리할 때와 전체 소요 시간이 크게 차이 나지 않습니다.
# =================================================================

# [입력 배치를 리스트로 준비]
# 여러 개의 문자열을 담은 파이썬 리스트(List)를 만듭니다.
prompts = [
    "Python의 장점 3가지를 알려주세요.",
    "머신러닝과 딥러닝의 차이점을 설명해주세요.",
    "좋은 프롬프트 엔지니어링의 원칙은 무엇인가요?",
    "Docker를 사용하는 이유는 무엇인가요?",
    "REST API와 GraphQL의 차이점을 설명해주세요."
]

# [추론 및 시간 측정]
start_time = time.time()  # 시작 시간 기록 (Unix Timestamp)

# llm.generate에 리스트(prompts)를 통째로 넘깁니다.
# [답변 생성 방식] 
# - 답변을 '하나씩' 순차적으로 생성하는 것이 아니라, 
# - GPU의 병렬 연산 능력을 사용하여 5개의 답변 데이터(Token)를 '동시에' 계산해냅니다.
outputs = llm.generate(prompts, sampling_params)

elapsed_time = time.time() - start_time  # 종료 시간 - 시작 시간 = 소요 시간

# [전체 통계 출력]
# 병렬 처리 덕분에 전체 소요 시간을 질문 개수로 나눈 '평균 시간'이 매우 짧게 나타납니다.
print(f"--- 총 처리 시간: {elapsed_time:.2f}초")
print(f"--- 프롬프트당 평균: {elapsed_time/len(prompts):.2f}초 (병렬 처리의 위력)")
print("=" * 60)  # 구분선 출력

# [개별 결과 확인]
# 결과 리스트(outputs)를 순회하며 각 질문과 생성된 답변의 앞부분을 확인합니다.
for i, output in enumerate(outputs):
    print(f"\n[{i+1}] 📝 {output.prompt}")  # 원본 질문 출력
    # 생성된 텍스트(output.outputs[0].text)의 앞 200자만 슬라이싱하여 출력
    print(f"- {output.outputs[0].text[:200]}...")
    print("-" * 60)

Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

--- 총 처리 시간: 1.23초
--- 프롬프트당 평균: 0.25초

[1] 📝 Python의 장점 3가지를 알려주세요.
-  Python의 장점은 다음과 같습니다:

1. 간결함: Python은 간결하고 직관적인 문법을 사용하여 코드를 작성할 수 있어, 개발자가 쉽게 코드를 작성하고 이해할 수 있습니다. 

2. 가독성: Python은 가독성을 위해 간결한 문법을 사용하여 코드를 작성할 수 있습니다. 이는 개발자가 코드를 이해하고 유지보수하는데 도움이 됩니다.

3. 확장성: ...
------------------------------------------------------------

[2] 📝 머신러닝과 딥러닝의 차이점을 설명해주세요.
-  머신러닝과 딥러닝은 모두 컴퓨터 알고리즘을 사용하여 데이터를 처리하고 분석하는 방법입니다. 그러나 두 방법의 차이점은 다음과 같습니다.

1. 구조: 딥러닝은 머신러닝의 일부로, 머신러닝이 더 일반적인 구조를 가진 반면, 딥러닝은 그 구조를 더 복잡하게 하여 더 많은 학습률을 가집니다.

2. 학습: 딥러닝은 여러 층의 신경망을 사용하여 학습을 수행하며,...
------------------------------------------------------------

[3] 📝 좋은 프롬프트 엔지니어링의 원칙은 무엇인가요?
-  그 원칙에 대해 설명해 주시면 감사하겠습니다.

대부분의 프롬프트 엔지니어링에 대해 잘 이해하고 있지만, 원칙에 대한 이해가 부족해요. 무엇을 더 설명해 주실 수 있나요? 

물론이지만, '프롬프트 엔지니어링'이란 무엇인가요? 프롬프트 엔지니어링이란 무엇이죠? 

프롬프트 엔지니어링은 프롬프트를 생성하고, 검증하고, 최적화하는 과정을 말합니다. 이는 자연...
------------------------------------------------------------

[4] 📝 Docker를 사용하는 이유는 무엇인가요?
-  Docker는 컴퓨터에서 실행되는 프로그램을 최적화하고 관

### 실습 2: 배치 크기와 처리량 실험

프롬프트 개수를 늘려가며 처리량 테스트

In [ ]:
# [배치 크기에 따른 성능 실험]
# =================================================================
# [이 셀의 핵심: GPU의 가성비(Efficiency) 테스트]
# 1. 목적: 질문을 1개 보낼 때와 20개 보낼 때, 시간이 얼마나 차이 나는지 측정합니다.
# 2. 가설: 하나씩 20번 하면 20배 오래 걸리겠지만, 배치는 동시에 처리하므로 시간이 거의 비슷할 것이다.
# 3. 지표: 
#    - 소요 시간(Elapsed Time): 전체 작업을 마치는 데 걸린 초.
#    - 처리량(Throughput): 1초당 처리된 요청 수(req/s). 높을수록 효율적입니다.
# =================================================================

# 기본 프롬프트 준비 (정확한 비교를 위해 짧고 동일한 질문 사용)
base_prompt = "대한민국의 수도는 어디인가요? 간단히 답해주세요."

# 짧은 답변을 유도하여 생성 시간 변수를 최소화 (max_tokens=50)
params = SamplingParams(temperature=0.7, max_tokens=50)

# 실험할 배치 크기 목록 (1개부터 20개까지 단계별 확장)
batch_sizes = [1, 5, 10, 20]
results = []  # 실험 데이터를 저장할 리스트

# 각 배치 크기별로 루프를 돌며 성능 측정
for batch_size in batch_sizes:
    # [리스트 곱셈] 동일한 질문을 배치 크기만큼 복사하여 리스트 생성
    # 예: batch_size가 5라면 ["수도는?", "수도는?", "수도는?", "수도는?", "수도는?"]
    prompts = [base_prompt] * batch_size

    start = time.time()  # 타이머 시작
    
    # llm.generate: 여기서 실제 GPU 병렬 연산이 일어납니다.
    # GPU 내부의 수천 개 코어가 배치 안의 질문들을 '동시에' 나누어 처리합니다.
    outputs = llm.generate(prompts, params) 
    
    elapsed = time.time() - start  # 전체 소요 시간 계산

    # [처리량(Throughput) 계산]
    # 공식: 전체 요청 수 / 걸린 시간 = 1초당 몇 개의 질문을 해치웠는가?
    throughput = batch_size / elapsed

    # 결과 저장 (나중에 차트로 그리거나 분석하기 위함)
    results.append((batch_size, elapsed, throughput))

    # 결과 출력: 2d(정수 2자리), .2f(소수점 2자리) 정렬
    print(f"배치 크기: {batch_size:2d} | 시간: {elapsed:.2f}초 | 처리량: {throughput:.2f} req/s")

print("\n" + "="*60)
print("--- [실험 결론: GPU 병렬 처리의 위력] ---")
print("1. 배치 크기가 1에서 20으로 늘어나도, 전체 시간은 거의 늘어나지 않습니다.")
print("2. 이는 GPU가 한 번에 많은 양의 데이터를 요리할 수 있는 '대용량 그릴'이기 때문입니다.")
print("3. 결과적으로 배치 크기를 키울수록 '초당 처리량(req/s)'은 폭발적으로 증가합니다.")
print("4. 즉, 서비스 운영 시에는 요청을 모아서 한꺼번에 처리하는 것이 훨씬 경제적입니다.")
print("="*60)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

배치 크기:  1 | 시간: 0.23초 | 처리량: 4.26 req/s


Adding requests:   0%|          | 0/5 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/5 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

배치 크기: 10 | 시간: 0.24초 | 처리량: 41.78 req/s


Adding requests:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

배치 크기: 20 | 시간: 0.24초 | 처리량: 83.71 req/s

--- !!! 배치 크기가 커져도 시간이 비례해서 증가하지 않는 것을 확인!!!!


---

## Part 3: OpenAI 호환 API 서버

vLLM은 OpenAI API와 호환되는 서버 제공: 기존 OpenAI 코드를 그대로 사용가능

### 3.1 기존 LLM 객체 정리

In [ ]:
# 메모리 확보를 위해 기존 모델 삭제
del llm
import gc
gc.collect()
torch.cuda.empty_cache()

print("--- 메모리 정리 완료")

--- 메모리 정리 완료


### 3.2 API 서버 백그라운드 실행

In [ ]:
# [vLLM OpenAI 호환 서버 백그라운드 실행]
# =================================================================
# [이 셀의 역할: 내 컴퓨터를 'ChatGPT API 서버'처럼 만들기]
# 1. OpenAI 호환: OpenAI에서 만든 라이브러리(SDK)를 그대로 가져다 쓸 수 있도록 
#    서버 규격을 똑같이 맞춰서 띄웁니다. (편의성 최강)
# 2. 백그라운드 실행: 쥬피터 셀 실행이 끝나도 서버가 종료되지 않고 
#    컴퓨터 내부적으로 계속 살아있게 만듭니다.
# 3. 로그 기록: 서버가 켜지는 과정이나 에러 내용을 실시간으로 파일에 적어둡니다.
# =================================================================

# !nohup: 'No HangUp'의 약자로, 현재 연결이 끊겨도(창을 닫아도) 프로그램을 죽이지 말라는 뜻
# python -m vllm.entrypoints.openai.api_server: vLLM 서버 모듈을 실행
# & (마지막 기호): "이 명령은 뒤(백그라운드)에서 실행하고, 나는 바로 다음 코드를 진행할게"라는 뜻
!nohup python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-1.5B-Instruct \
    --dtype half \
    --port 8000 \
    --gpu-memory-utilization 0.75 \
    --max-model-len 2048 \
    > /tmp/vllm_server.log 2>&1 &

# [명령어 뒷부분 특수 기호 설명]
# > /tmp/vllm_server.log: 서버가 출력하는 모든 메시지를 이 파일에 저장해라
# 2>&1: 에러 메시지(2)도 일반 메시지(1)가 들어가는 곳(파일)에 합쳐서 같이 저장해라

# [서버 설정 옵션 설명]
# --model: 서버에 올릴 AI 모델의 이름 혹은 경로
# --dtype half: 성능 저하를 최소화하면서 메모리 사용량을 절반으로 줄임
# --port 8000: 'http://localhost:8000' 주소로 요청을 주고받겠다는 약속
# --gpu-memory-utilization 0.75: GPU 메모리의 75%를 서버가 독점해서 쓰겠다는 설정
# --max-model-len 2048: 입력과 답변을 합쳐 최대 2048토큰까지만 허용

print("--- 서버 시작 명령 완료!")
print("--- [주의] 서버가 실제로 다 켜지기까지(모델 로딩) 1~2분 정도 기다려야 합니다.")
print("--- 서버 로그 확인법: !tail -n 20 /tmp/vllm_server.log")

--- 서버 시작 중... 모델 로딩까지 약 1-2분 소요


In [ ]:
# [서버 시작 대기 및 상태 확인]
# =================================================================
# [이 셀의 역할: 서버가 완전히 켜졌는지 끈기 있게 확인하기]
# 1. 헬스 체크(Health Check): 서버가 정상인지 확인하는 전용 주소(/health)에 
#    계속 신호를 보내서 대답이 오는지 확인합니다.
# 2. 자동화된 대기: 사람이 일일이 기다릴 필요 없이, 컴퓨터가 알아서 
#    최대 120초(2초 x 60번) 동안 서버의 상태를 체크합니다.
# 3. 진행 상황 표시: 기다리는 동안 지루하지 않게 마침표(.)를 찍어 진행 중임을 알립니다.
# =================================================================

import time

print("--- 서버 준비 대기 중 (최대 2분 소요)...")

# 최대 60번 반복 (2초씩 쉬므로 총 120초 동안 시도)
for i in range(60):
    time.sleep(2)  # 2초간 일시 정지 (서버에게 숨 돌릴 시간 주기)
    
    # [상태 확인 명령어 실행]
    # subprocess: 파이썬 내부에서 리눅스 명령어(curl)를 실행하게 해주는 모듈
    # curl -s ... /health: 서버의 '건강한지' 물어보는 전용 통로에 조용히(-s) 신호를 보냄
    import subprocess
    result = subprocess.run(['curl', '-s', 'http://localhost:8000/health'], capture_output=True)
    
    # [성공 여부 판단]
    # returncode == 0: 리눅스에서 명령어 실행이 '성공'했음을 의미 (즉, 서버가 대답을 주기 시작함)
    if result.returncode == 0:
        print(f"\n--- 서버 준비 완료! (약 {(i+1)*2}초 소요되었습니다.)")
        break  # 대답이 왔으므로 더 기다리지 않고 루프 탈출
    
    # 아직 대답이 없으면 마침표(.)를 찍고 다시 루프를 돔
    # flush=True: 마침표를 찍자마자 바로 화면에 보여주도록 강제함
    print(".", end="", flush=True)

# 만약 60번 다 돌았는데도 대답이 없는 경우 (else는 루프가 break 없이 끝났을 때만 실행됨)
else:
    print("\n" + "="*60)
    print("--- [경고] 서버 시작이 너무 오래 걸리고 있습니다.")
    print("--- 1. GPU 메모리가 부족한지 확인하세요.")
    print("--- 2. !tail -n 20 /tmp/vllm_server.log 명령어로 로그를 확인해보세요.")
    print("="*60)

--- 서버 준비 대기 중...
................................
--- 서버 준비 완료! (66초 소요)


In [ ]:
# # 서버 로그 확인 (문제 발생시 디버깅용)
# !tail -30 /tmp/vllm_server.log

### 3.3 사용 가능한 모델 확인

In [ ]:
!curl -s http://localhost:8000/v1/models | python -m json.tool

{
    "object": "list",
    "data": [
        {
            "id": "Qwen/Qwen2.5-1.5B-Instruct",
            "object": "model",
            "created": 1769424421,
            "owned_by": "vllm",
            "root": "Qwen/Qwen2.5-1.5B-Instruct",
            "parent": null,
            "max_model_len": 2048,
            "permission": [
                {
                    "id": "modelperm-b3f49c390ff1230f",
                    "object": "model_permission",
                    "created": 1769424421,
                    "allow_create_engine": false,
                    "allow_sampling": true,
                    "allow_logprobs": true,
                    "allow_search_indices": false,
                    "allow_view": true,
                    "allow_fine_tuning": false,
                    "organization": "*",
                    "group": null,
                    "is_blocking": false
                }
            ]
        }
    ]
}


### 3.4 OpenAI 클라이언트로 API 호출

In [ ]:
from openai import OpenAI

# vLLM 서버에 연결
client = OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy"  # vLLM은 API 키 검증 없음
)

print("--- OpenAI 클라이언트 연결 완료")

--- OpenAI 클라이언트 연결 완료


In [ ]:
# [API 호출]
# client.chat.completions.create 메서드를 사용하여 서버에 '채팅 완료' 요청을 보냅니다.
# 이는 HTTP POST 요청을 래핑(Wrapping)한 함수입니다.
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",  # 호출할 모델 이름 (서버에 로드된 모델과 일치해야 함)
    messages=[  # 대화 내역 리스트 (List of Dictionaries)
        {"role": "system", "content": "당신은 친절하고 유능한 AI 어시스턴트입니다."},  # 시스템 프롬프트(AI의 역할 부여)
        {"role": "user", "content": "서울에서 부산까지 여행 계획을 세워줘. 2박 3일 일정으로."}  # 사용자의 질문
    ],
    temperature=0.7,  # 창의성 조절
    max_tokens=500    # 최대 응답 길이
)

# [응답 처리]
# response 객체는 서버로부터 받은 JSON 응답을 파이썬 객체로 변환한 것입니다.
# response.choices[0].message.content 에 AI의 답변 텍스트가 들어있습니다.
print("--- 응답:")
print(response.choices[0].message.content)

--- 응답:
네, 서울에서 부산까지의 2박 3일 여행 계획을 도와드릴게요. 아래는 예상적인 일정과 추천行程입니다:

Day 1: 
- 오전 8시: 서울에 도착 후, 출발지점에서 호텔로 이동.
- 오후 4시: 호텔에서 편안히 식사享用합니다.
- 저녁: 서초구의 '제주도식 전통 음식'을 즐길 수 있는 '제주도 맛집'을 방문.

Day 2:
- 아침: 부산항에서 첫 번째 목적지인 해운대를 방문하여 해변을 걸으며 시간 보내세요.
- 점심: 해운대 근처의 '해운대 맛집'을 방문하세요.
- 오후: 부산역에서 타고 인근의 대형 슈퍼마켓으로 향하여 물품들을 구입하는 것이 좋습니다.
- 저녁: 부산의 한 곳에서 '부산의 본향'을 즐기며 가족이나 친구들과 함께 나누는 시간을 갖는 것이 좋습니다.

Day 3:
- 아침: 부산역에서 체류 후, 다음 목적지인 사하구를 방문합니다.
- 점심: 사하구의 '사하구 맛집'을 방문하세요.
- 오후: 사하구에 위치한 공원을 찾고 자연을 느낄 기회를 갖는 것이 좋습니다.
- 저녁: 부산의 한 곳에서 '부산의 본향'을 즐기며 가족이나 친구들과 함께 나누는 시간을 갖는 것이 좋습니다.

이런 일정은 일반적인 추천이며, 개인의 취미나 관심 분야에 따라 조금씩 조정하실 수 있습니다. 또한, 날씨 및 교통 상황 등도 고려해야 하니 참고하시기 바랍니다.


In [ ]:
# 응답 메타데이터 확인
print(f"모델: {response.model}")
print(f"토큰 사용량: {response.usage}")
print(f"종료 이유: {response.choices[0].finish_reason}")

모델: Qwen/Qwen2.5-1.5B-Instruct
토큰 사용량: CompletionUsage(completion_tokens=430, prompt_tokens=54, total_tokens=484, completion_tokens_details=None, prompt_tokens_details=None)
종료 이유: stop


### 3.5 스트리밍 응답

In [ ]:
# [스트리밍(Streaming) 요청]
# stream=True 옵션을 주면, 서버는 답변이 완성될 때까지 기다리지 않고
# 생성되는 토큰(글자) 단위로 즉시 데이터를 보내줍니다. (마치 타자 치듯)
print("--- 스트리밍 응답:\n")

stream = client.chat.completions.create(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    messages=[
        {"role": "user", "content": "Python으로 피보나치 수열을 계산하는 함수를 작성하고 설명해줘."}
    ],
    temperature=0.7,
    max_tokens=400,
    stream=True  # [핵심] 스트리밍 모드 활성화
)

# [제너레이터(Generator) 반복]
# stream 변수는 '제너레이터' 객체입니다. for문을 돌 때마다 서버에서 오는 데이터 조각(chunk)을 하나씩 꺼냅니다.
for chunk in stream:
    # chunk.choices[0].delta.content: 이번 조각에 새로 추가된 텍스트가 있는지 확인
    # delta는 '변화량'을 의미합니다. (이전 청크 대비 새로 생긴 글자)
    if chunk.choices[0].delta.content:
        # print 함수 옵션:
        # end="": 줄바꿈(\n)을 하지 않고 이어서 출력
        # flush=True: 출력 버퍼를 강제로 비워서 즉시 화면에 표시되게 함 (이게 없으면 모아서 뜸)
        print(chunk.choices[0].delta.content, end="", flush=True)

print("\n\n--- 스트리밍 완료")

--- 스트리밍 응답:

Python에서 피보나치 수열을 계산하는 가장 간단한 방법은 재귀 함수를 사용할 수 있습니다. 재귀는 호출 자기 자신을 반복적으로 실행하는 방식입니다. 하지만 이 방법은 메모리와 시간 복잡도가 상당히 높아서 큰 n 값에서는 성능이 좋지 않을 수 있습니다.

다음은 Python에서 재귀적으로 피보나치 수열을 계산하는 함수입니다:

```python
def fibonacci(n):
    if n <= 1:
        return n
    else:
        return (fibonacci(n-1) + fibonacci(n-2))
```

이 함수는 피보나치 수열의 n 번째 항을 반환합니다. n이 0 또는 1인 경우 그 자체로 반환하며, 그렇지 않으면 이전 두 항에 대한 합으로 계산됩니다.

주의해야 할 점은 피보나치 수열은 매우 긴 숫자를 가질 때마다 시간과 메모리를 많이 소비하기 때문에, 특히 큰 n 값일 때에는 일반적인 반복문보다는 재귀함수를 사용하지 않는 것이 좋습니다. 또한, 재귀호출은 최대 1000회까지 가능하므로, 너무 큰 n 값을 사용하면 에러가 발생할 수 있습니다.

--- 스트리밍 완료


### 실습 3-1: 다중 턴 대화

대화 기록을 유지하며 연속 대화 실행

In [ ]:
# [멀티턴(Multi-turn) 대화 구현: AI에게 '기억'을 선물하는 법]
# =================================================================
# [이 셀의 핵심 원리: 누적형 대화 관리 (Cumulative History)]
#
# 1. LLM의 한계 (Stateless): 
#    - AI 모델은 한 번의 요청이 끝나면 방금 무슨 말을 했는지 완전히 잊어버립니다.
#    - "이전 내용 기억해?"라고 물으면 "무슨 대화요?"라고 답하는 게 정상입니다.
#
# 2. 해결책 (History Injection): 
#    - AI가 기억하게 만드는 유일한 방법은 '과거의 모든 대화 내용'을 
#    - 매 요청(Request) 때마다 리스트에 담아 통째로 다시 보내주는 것입니다.
#
# 3. 리스트 관리 방식 (Append, Not Replace):
#    - 기존 내용을 지우고 덮어쓰는 것이 아닙니다. 
#    - [시스템 설정] -> [사용자 질문1] -> [AI 답변1] -> [사용자 질문2] -> [AI 답변2]...
#    - 위와 같이 리스트의 끝에 새로운 메시지를 계속 이어 붙여(Append) 길이를 늘려갑니다.
#
# 4. 역할 분담 (Roles):
#    - System: 대화의 게임 법칙 설정 (친절한 튜터가 되어라 등)
#    - User: 실제 사용자가 던지는 질문
#    - Assistant: AI가 생성한 답변 (다음 질문 시 문맥 파악을 위해 필요)
# =================================================================

# 1. 대화 기록을 저장할 리스트 초기화 (이 리스트가 AI의 '단기 기억 장소'가 됩니다.)
conversation_history = [
    # [중요] 시스템 메시지는 항상 리스트의 0번 인덱스(맨 앞)에 고정되어야 합니다.
    # 대화가 길어져도 AI가 '자신이 누구인지(튜터)' 잊지 않게 중심을 잡아줍니다.
    {"role": "system", "content": "당신은 Python 튜터입니다. 친절하게 설명해주세요."}
]

# 2. 대화 관리 함수 정의: 사용자 메시지를 받고, 기억을 업데이트하고, 답변을 반환합니다.
def chat(user_message):
    # [STEP 1: 사용자의 새 메시지 기록]
    # 리스트의 가장 마지막에 사용자가 지금 입력한 말을 추가합니다.
    # 예: "리스트 차이가 뭐야?"라는 문자열이 {"role": "user", "content": "..."} 객체로 변해 추가됨
    conversation_history.append({"role": "user", "content": user_message})

    # [STEP 2: 전체 기억을 담아 API 호출]
    # 여기서 핵심은 `messages=conversation_history`입니다.
    # 단일 질문이 아닌, 시스템 설정부터 방금 추가한 질문까지의 '전체 리스트'를 서버로 전송합니다.
    # 이제 AI는 이 리스트를 처음부터 쭉 읽으며 문맥(Context)을 파악한 뒤 답변을 생성합니다.
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=conversation_history,  # <--- 이 리스트가 길어질수록 AI의 기억력도 깊어집니다.
        temperature=0.7,
        max_tokens=300
    )

    # [STEP 3: 생성된 AI 답변 추출]
    # 서버로부터 전달받은 복잡한 응답 객체에서 실제 글자(text) 부분만 골라냅니다.
    assistant_message = response.choices[0].message.content

    # [STEP 4: AI의 답변도 기억에 추가]
    # 다음 턴(Turn)의 대화를 위해 AI가 방금 한 말도 리스트 끝에 추가합니다.
    # 이렇게 해야 다음에 사용자가 "방금 네가 한 말 무슨 뜻이야?"라고 물을 때 AI가 대답할 수 있습니다.
    conversation_history.append({"role": "assistant", "content": assistant_message})

    # 사용자에게 보여줄 최종 답변 텍스트를 반환합니다.
    return assistant_message

# [테스트: 대화의 연속성 확인]

# 첫 번째 질문 (주제 시작)
# 순서: [System] -> [User 1] -> (AI 계산) -> [Assistant 1] 저장
print("- User: 리스트와 튜플의 차이점이 뭐야?")
print(f"--- Assistant: {chat('리스트와 튜플의 차이점이 뭐야?')}\n")

# 두 번째 질문 ("그럼" 이라는 지시어 사용)
# 순서: [System] -> [User 1] -> [Assistant 1] -> [User 2] -> (AI 계산) -> [Assistant 2] 저장
# AI는 [User 1]과 [Assistant 1] 내용을 읽었기 때문에 "그럼"이 리스트/튜플을 뜻함을 압니다.
print("- User: 그럼 언제 튜플을 써야 해?")
print(f"--- Assistant: {chat('그럼 언제 튜플을 써야 해?')}\n")

# 세 번째 질문 (코드 예시 요청)
# 순서: [System] -> ... -> [User 3] -> (AI 계산) -> [Assistant 3] 저장
# 질문에 '무슨 코드'인지 명시하지 않아도 앞선 대화 문맥상 리스트/튜플 코드를 보여주게 됩니다.
print("- User: 예제 코드 보여줘")
print(f"--- Assistant: {chat('예제 코드 보여줘')}")

- User: 리스트와 튜플의 차이점이 뭐야?
--- Assistant: Python에서 리스트와 튜플은 두 가지 유사한 자료형을 제공하나, 서로 다른 특징과 활용 방식이 있습니다.

1. **구조적 구분**:
   - **리스트 (list)**: 
     1.0.1) 원소를 순서대로 관리합니다.
     1.0.2) 원소가 변경될 수 있음.
     1.0.3) 다양한 데이터 타입으로 이루어져 있습니다.
     예: `my_list = [1, 'two', True]`

   - **튜플 (tuple)**:
     2.0.1) 원소는 순서대로 관리됩니다.
     2.0.2) 원소는 변경되지 않음.
     2.0.3) 모든 요소는 고정된 데이터 타입만 가능합니다.
     예: `my_tuple = (1, 'two', True)`

2. **변경 가능성**:
   - **리스트**: 원소가 변경 가능합니다.
   - **튜플**: 원소는 변경되지 않으며, 이를 통해 더 효율적인 코드 작성 가능.

3. **주요 사용 사례**:
   - **리스트**: 연속된 데이터를 관리하고 변경할 때 많이 사용됩니다.
   - **튜플**: 고정된 데이터로 구성되어 있으며, 데이터를 전달하거나 특정

- User: 그럼 언제 튜플을 써야 해?
--- Assistant: 튜플을 사용하는 가장 좋은 이유는 그들이 불변성을 유지한다는 것입니다. 이는 특히 데이터베이스 연결 정보나 API 헤더 등의 경우에 유용합니다. 이러한 정보들은 변경될 일이 없기 때문에 튜플로 저장하면 안전합니다.

예를 들어, 다음과 같이 API 헤더를 저장할 때 튜플을 사용하는 것이 좋습니다:

```python
import requests

headers = ('Content-Type', 'application/json')
response = requests.get('https://example.com/api/data', headers=headers)
```
위 코드에서는 `headers`라는 이

### 실습 3-2: 동시 요청 처리

여러 요청을 동시에 보내 vLLM의 배치 처리 성능을 확인

In [ ]:
import concurrent.futures  # 동시성(Concurrency) 처리를 위한 표준 라이브러리 (여러 일을 한꺼번에 시킬 때 사용)
import time

# =================================================================
# [이 셀의 핵심: 멀티 스레딩(Multi-threading)을 이용한 서버 부하 테스트]
#
# 1. 동시성(Concurrency): 10명의 사용자가 한꺼번에 서버에 접속하는 상황을 재현합니다.
# 2. ThreadPoolExecutor: 10개의 '가상 일꾼(Thread)'을 생성하여 각각 API 요청을 보내게 합니다.
# 3. vLLM의 강점: 서버(vLLM)는 이 10개의 요청을 받는 즉시 'Continuous Batching' 기술을 통해
#    GPU가 10개를 동시에 병렬로 처리하게 만듭니다. 
# 4. 결과: 10개를 하나씩 따로 보낼 때보다 전체 처리 시간이 압도적으로 단축됩니다.
# =================================================================

# [단일 요청 처리 함수: 한 명의 일꾼이 수행할 작업]
# 하나의 프롬프트를 받아 서버에 던지고, 답변이 올 때까지 기다린 뒤 정보를 기록합니다.
def make_request(prompt):
    # 각 요청별로 정확한 응답 시간을 측정하기 위해 시작 시간 기록
    start = time.time()
    
    # [동기식 호출의 활용] 
    # client.chat.completions.create는 원래 답변이 올 때까지 프로그램이 멈추는 '동기' 방식입니다.
    # 하지만 이 함수를 '스레드(Thread)' 안에서 실행하면, 이 일꾼이 기다리는 동안 
    # 다른 일꾼들이 자기 할 일을 할 수 있어 전체적으로는 '동시'에 진행되는 것처럼 보입니다.
    response = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100
    )
    
    elapsed = time.time() - start # 이 일꾼이 답변을 받는 데 걸린 시간
    
    # 나중에 통계를 내기 위해 (질문 요약, 소요시간, 답변 글자수)를 바구니에 담아 반환
    return prompt[:30], elapsed, len(response.choices[0].message.content)

# 실험을 위해 서로 다른 10개의 가벼운 질문 리스트를 준비합니다.
prompts = [
    "1+1은?", "Python의 창시자는?", "HTTP와 HTTPS의 차이점?", "Docker란 무엇인가?",
    "Git의 장점 3가지", "REST API란?", "SQL과 NoSQL 비교",
    "머신러닝 종류 3가지", "클라우드 컴퓨팅이란?", "DevOps가 뭐야?"
]

# [동시 요청 실행 단계]
print("=========== [실험 시작] 10개 요청을 10명의 일꾼이 동시에 전송 중...\n")
total_start = time.time() # 전체 실험 시작 시간

# ThreadPoolExecutor: '일꾼들의 대기소'를 관리하는 관리자 객체입니다.
# max_workers=10: "지금부터 일꾼 10명을 고용해서 대기시켜라"라는 뜻입니다. (질문이 10개이므로 1:1 매칭)
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    # executor.map(함수, 데이터리스트):
    # 1. 데이터리스트(prompts)에서 질문 10개를 꺼내 일꾼 10명에게 하나씩 나눠줍니다.
    # 2. 모든 일꾼은 동시에 'make_request'를 실행하며 서버에 질문을 던집니다.
    # 3. 결과를 반환받은 순서가 제각각이더라도, map 함수가 똑똑하게 '입력 순서대로' 다시 줄을 세워줍니다.
    results = list(executor.map(make_request, prompts))

# 모든 일꾼이 임무를 마치고 'with' 문을 빠져나오면 전체 종료 시간을 기록합니다.
total_elapsed = time.time() - total_start

# [결과 리포트 출력]
print(f"{'프롬프트(질문)':<35} {'개별 응답시간':>15} {'응답길이':>10}")
print("=" * 70)
for prompt, elapsed, length in results:
    # 각 질문들이 거의 비슷한 시간대에 응답을 받았음을 확인할 수 있습니다.
    print(f"{prompt:<35} {elapsed:>13.2f}초 {length:>8}자")

print("=" * 70)
print(f"\n--- [실험 결과 분석] ---")
print(f"1. 전체 소요 시간: {total_elapsed:.2f}초")
print(f"   (하나씩 10번 했다면 약 {sum(r[1] for r in results):.2f}초가 걸렸을 것입니다.)")

# 처리량(Throughput) = 전체 질문 수 / 전체 소요 시간
# vLLM 서버의 '동시 처리 능력'이 좋을수록 이 숫자가 높아집니다.
print(f"2. 서버 처리량(Throughput): {len(prompts)/total_elapsed:.2f} requests/sec")
print(f"3. 결론: vLLM은 여러 사용자가 동시에 요청을 보내도 '줄 서서 기다리게' 하지 않고")
print(f"   GPU의 여유 자원을 활용해 그 요청들을 한꺼번에 묶어서(Batch) 순식간에 해치웁니다.")

=========== 10개 요청 동시 전송 중...

프롬프트                                      응답시간       응답길이
1+1은?                                   0.07초        7자
Python의 창시자는?                           0.55초      195자
HTTP와 HTTPS의 차이점?                       0.55초      194자
Docker란 무엇인가?                           0.54초      186자
Git의 장점 3가지                             0.55초      179자
REST API란?                              0.55초      206자
SQL과 NoSQL 비교                           0.55초      184자
머신러닝 종류 3가지                             0.54초      202자
클라우드 컴퓨팅이란?                             0.53초      184자
DevOps가 뭐야?                             0.54초      182자

--- 총 소요 시간: 0.56초
--- 처리량: 17.94 requests/sec



## Part 4: 정리

### 4.1 서버 종료

In [ ]:
# vLLM 서버 프로세스 종료
!pkill -f "vllm.entrypoints"
print("vLLM 서버 종료 완료")

vLLM 서버 종료 완료


### 정리

1. **vLLM 특징**
   - PagedAttention으로 메모리 효율 최적화
   - Continuous Batching으로 처리량 극대화
   - OpenAI API 호환으로 기존 코드 재사용 가능

2. **Offline Inference**
   - `LLM` 클래스로 직접 추론
   - `SamplingParams`로 생성 파라미터 제어
   - 배치 추론으로 효율적 처리

3. **OpenAI 호환 API**
   - `/v1/chat/completions` 엔드포인트
   - 스트리밍 지원
   - 기존 OpenAI 클라이언트 그대로 사용

---

### 혼자해보기 과제

1. 다른 모델 (`google/gemma-2-2b-it`)로 동일 실험 반복
2. `top_k` 파라미터 추가하여 샘플링 실험



# 📝 [혼자 해보기] 과제

앞서 배운 vLLM의 기능을 활용하여 직접 실험해보는 단계입니다.

### 과제 목표
1.  **다른 모델 사용해보기**: Qwen 대신 구글의 `google/gemma-2-2b-it` 모델을 로드하여 속도와 한국어 성능을 비교해봅니다.
2.  **샘플링 파라미터 튜닝**: `top_k` 파라미터를 조절하여 AI의 답변 스타일이 어떻게 변하는지 확인합니다.

In [ ]:
# [0. 메모리 초기화]
# 새로운 모델(Gemma)을 로드하려면 기존에 GPU를 잡고 있는 프로세스를 정리해야 합니다.
# 만약 이 코드를 실행하지 않으면 GPU 메모리 부족(OOM) 오류가 발생할 수 있습니다.

import gc
import torch

# 1. vLLM 엔진 객체 삭제 (만약 정의되어 있다면)
try:
    del llm
except NameError:
    pass

# 2. 파이썬 가비지 컬렉터 실행 & CUDA 캐시 비우기
gc.collect()
torch.cuda.empty_cache()

# 3. 떠있는 API 서버가 있다면 강제 종료
!pkill -f "vllm.entrypoints"

print("--- 🧹 메모리 청소 완료! 새로운 모델을 맞이할 준비가 되었습니다.")

--- 🧹 메모리 청소 완료! 새로운 모델을 맞이할 준비가 되었습니다.


In [ ]:
# [과제 1: Google Gemma-2-2b-it 모델 로드 및 테스트]
# Qwen이 아닌 다른 아키텍처의 모델을 vLLM으로 구동해봅니다.
# Gemma는 구글에서 만든 고성능 경량 모델입니다.

from vllm import LLM, SamplingParams

print("--- Gemma 모델 로딩 시작...")

# 1. 모델 로드
# 주의: Gemma 모델은 HuggingFace에서 사용 동의(Gate Access)가 필요할 수 있습니다.
# 에러 발생 시 HuggingFace 토큰이 올바르게 설정되었는지 확인하세요.
llm_gemma = LLM(
    model="google/gemma-2-2b-it",
    dtype="bfloat16",                 # 메모리 효율을 위해 bf16 사용
    gpu_memory_utilization=0.9,   # 모델이 작으므로 메모리를 조금 덜 잡아도 됩니다.
    max_model_len=4096
)

# 2. 테스트 프롬프트
prompt = "인공지능의 미래에 대해 한 문장으로 정의해줘."

# 3. 추론 실행
output = llm_gemma.generate([prompt], SamplingParams(temperature=0.7, max_tokens=100))

print(f"\n📝 질문: {prompt}")
print(f"💡 Gemma의 답변: {output[0].outputs[0].text}")

--- Gemma 모델 로딩 시작...
INFO 01-26 10:49:55 [utils.py:263] non-default args: {'dtype': 'bfloat16', 'max_model_len': 4096, 'disable_log_stats': True, 'model': 'google/gemma-2-2b-it'}
INFO 01-26 10:49:56 [model.py:530] Resolved architecture: Gemma2ForCausalLM
INFO 01-26 10:49:56 [model.py:1545] Using max model len 4096
INFO 01-26 10:49:56 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.


tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

INFO 01-26 10:51:15 [llm.py:347] Supported tasks: ['generate']


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]


📝 질문: 인공지능의 미래에 대해 한 문장으로 정의해줘.
💡 Gemma의 답변: 

## AI의 미래: **인공지능은 우리 삶을 더 쉽고 빠르게 만들어줄 핵심적인 도구이며, 윤리적이고 책임 있는 개발과 적용을 통해 더 나은 미래를 위한 발전의 중심 역할을 맡을 것이다.** 




In [ ]:
# [과제 2: Top-k 샘플링 파라미터 실험]
# Top-k는 모델이 다음 단어를 선택할 때, 확률이 높은 상위 K개의 후보 중에서만 고르는 방식입니다.
# Top-p(누적 확률)와는 다르게 '등수'로 딱 자르기 때문에 더 예측 가능한 결과를 얻을 때 유리할 수 있습니다.

prompts = ["재미있는 판타지 소설 제목 5개만 지어줘."]

# 1. 실험 A: Top-k = 1 (Greedy Search와 유사)
# 무조건 가장 확률이 높은 1등 단어만 선택 -> 매우 뻔하고 고정된 답변이 나옴
params_strict = SamplingParams(temperature=1.0, top_k=1, max_tokens=150)

# 2. 실험 B: Top-k = 10 (균형 잡힌 선택)
# 상위 10개 정도면 문맥에 맞는 적절한 단어들 내에서 변화를 줍니다.
params_balanced = SamplingParams(temperature=1.0, top_k=10, max_tokens=150)

# 3. 실험 C: Top-k = 50 (다양성 허용)
# 상위 50개 단어 중에서 랜덤 선택 -> 좀 더 창의적이거나 엉뚱한 답변이 나올 수 있음
params_creative = SamplingParams(temperature=1.0, top_k=50, max_tokens=150)

print("--- 🧪 실험 A: Top-k=1 (엄격/건조함) ---")
output_strict = llm_gemma.generate(prompts, params_strict)
print(output_strict[0].outputs[0].text)

print("\n--- 🧪 실험 B: Top-k=10 (적절한 균형) ---")
output_balanced = llm_gemma.generate(prompts, params_balanced)
print(output_balanced[0].outputs[0].text)

print("\n--- 🧪 실험 C: Top-k=50 (창의적/다양함) ---")
output_creative = llm_gemma.generate(prompts, params_creative)
print(output_creative[0].outputs[0].text)

--- 🧪 실험 A: Top-k=1 (엄격/건조함) ---


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]



1. **빛나는 숲의 비밀**
2. **마법의 깃털, 꿈의 폭풍**
3. **사랑의 빛, 검은 밤의 춤**
4. **영혼의 울음, 숨결의 춤**
5. **마법의 숲, 잊혀진 왕자**


## 판타지 소설 제목 5개 

1. **빛나는 숲의 비밀**
2. **마법의 깃털, 꿈의 폭풍**
3. **사랑의 빛, 검은 밤의 춤**
4. **영혼

--- 🧪 실험 B: Top-k=10 (적절한 균형) ---


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]



## 좋아할 만한 재미있는 판타지 소설 제목 5개

1. **마법과 우정의 밤하늘:** 푸른빛 마법으로 꿈을 꾸고, 친구들과 춤을 춰, 행복의 밤하늘을 향해 달려가는 로맨틱 판타지 소설 
2. **사막의 거인:** 바람이 춤추는 사막, 거친 땅을 뚫고 나서는 거인 이야기에 얽히는 웅장한 판타지를 담았습니다.
3. **반짝이는 숲의 밤:** 숲속 빛나는 흔

--- 🧪 실험 C: Top-k=50 (창의적/다양함) ---


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]



1. 빛과 사막의 장갑에 손가락 내리다
2. 너는 춤이 너의 능력이다
3. 바람의 약속
4. 고행 무한대의 낙엽
5. 불타는 뱀의 메시지

**게임 디자인 텍스트**

**장의 디자인:**

- **고어도**: 빛깔은 청록빛색으로, 검은색 눈통으로 탐색하는 듯한 느낌을 주지만 무의식적으로 뛰어난 재능을 지녔다고  설화된다.
- **신전**: 반짝이는 빛깔은 밤하


### [심화 과제] 파라미터 칵테일 (Mixing Parameters)

`top_k` 하나만 조절하는 것은 반쪽짜리 실험입니다. 실제 현업에서는 **Temperature(온도)**, **Top-p**, **Top-k**를 적절히 섞어서 최적의 답변을 찾아냅니다.

*   **순서**: 보통 `top_k`로 이상한 단어를 1차로 쳐내고 -> `top_p`로 그 중에서 괜찮은 후보군을 추리고 -> `temperature`로 최종 선택의 다양성을 줍니다.
*   **목표**: 파라미터를 섞었을 때 답변 품질이 어떻게 달라지는지 확인합니다.

In [ ]:
# [과제 3: 파라미터 조합(Cocktail) 실험]
# =================================================================
# [핵심 개념: LLM의 '품질'을 결정하는 3총사 비유적으로 이해하기]
#
# 1. Temperature (온도/창의성): "얼마나 모험을 할 것인가?"
#    - 높으면(1.2): 엉뚱하고 창의적인 답변. 하지만 너무 높으면 헛소리(Hallucination)를 합니다.
#    - 낮으면(0.2): 정답에 가까운 말만 함. 하지만 너무 낮으면 기계처럼 똑같은 말만 반복합니다.
#
# 2. Top-K (순위 컷오프): "상위 몇 등까지만 후보에 넣을 것인가?" (정확도 필터)
#    - 예: Top-K=50이면 가장 확률이 높은 단어 50개만 딱 추리고 나머지는 아예 버립니다.
#    - 역할: 이상한 단어가 아예 끼어들지 못하게 하는 '1차 차단벽'입니다.
#
# 3. Top-P (누적 확률 컷오프): "상위 몇 %의 가능성 안에서 놀 것인가?" (동적 필터)
#    - 예: Top-P=0.9이면 확률을 합쳐서 90%가 될 때까지의 단어들만 후보로 삼습니다.
#    - 역할: 상황에 따라 후보 숫자를 조절합니다. 확실한 상황에선 좁게, 애매한 상황에선 넓게!
# =================================================================

prompt = "판타지 소설의 첫 문장을 아주 매력적으로 써줘."

# 1. Temperature만 높게 줌 (제어 장치가 없는 상태)
# -> 창의성은 폭발하지만, 상위권 밖의 '이상한 단어'까지 마구잡이로 선택할 수 있습니다.
# -> 결과적으로 문장이 산으로 가거나 말이 안 되는 소리를 할 확률이 매우 높습니다.
params_wild = SamplingParams(temperature=1.2, max_tokens=100)

# 2. 황금 칵테일 조합 (창의성 + 안전장치 2개)
params_best_mix = SamplingParams(
    temperature=1.2,  # [창의성] 일단 아이디어는 대담하게 던져보되,
    top_k=50,         # [안전장치1] 아무리 대담해도 하위권 단어(이상한 말)는 후보에서 빼고,
    top_p=0.9,        # [안전장치2] 문맥상 충분히 발생 가능한(상위 90%) 단어들 중에서만 골라라!
    max_tokens=100
)

print("--- 🤪 제어 없는 야생의 답변 (Temperature=1.2) ---")
# 제어 장치가 없으면 단어들 사이의 개연성이 떨어질 수 있습니다.
output_wild = llm_gemma.generate([prompt], params_wild)
print(output_wild[0].outputs[0].text)

print("\n--- 🍸 황금 비율 칵테일 답변 (Temp=1.2 + Top-k=50 + Top-p=0.9) ---")
# 높은 온도로 창의성을 챙기면서, Top-K/P로 '선 넘는 단어'들을 걸러내어 '창의적이면서도 읽히는' 문장이 나옵니다.
output_mix = llm_gemma.generate([prompt], params_best_mix)
print(output_mix[0].outputs[0].text)

# [요약]
# - Top-K: '후보자 수'를 제한해서 완전 뜬금없는 단어를 막음 (정확도 확보)
# - Top-P: '확률 총합'을 제한해서 문맥상 자연스러운 범위로 한정 (유연한 정확도)
# - 이 둘을 Temperature와 섞어야 '똑똑하면서도 재미있는' 답변이 나옵니다!

--- 🤪 제어 없는 야생의 답변 (Temp=1.2) ---


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]



 '\'당신이 생각나기 어려운 곳에 나있는 정체없더라는 맨 claimed the,'
 

**Grammar:** Here's the breakdown of the grammar and why it works:

* **"' "` :** Includes the direct quotation marks. Some vogue use ellipses,...
* **"당신이 생각나기 어려운 곳에 나있는" :** 
    * **"신중하게 생각되게 둔 바늘

--- 🍸 황금 비율 칵테일 답변 (Temp=1.2 + Top-k=50 + Top-p=0.9) ---


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]



The wind whispered secrets through the ancient trees, its voice a low rumble echoing in the stillness of the forest.

You could just barely make out the flickering outline of a figure on a distant platform.

A chillingly beautiful woman, her red silk gown rustling like the whisper of a cold wind, turned her back on the shadows to face the dawn.

What is it about the first lines that make them so effective? 
* Firstly, the reader senses a sense of mystery and magic


---

## (참고) Troubleshooting

### GPU 메모리 부족 오류
```python
# 해결방법 1: 더 작은 모델 사용
model = "Qwen/Qwen2.5-0.5B-Instruct"

# 해결방법 2: GPU 메모리 사용량 줄이기
gpu_memory_utilization = 0.8
```

### 서버가 시작되지 않을 때
```bash
# 로그 확인
!cat /tmp/vllm_server.log

# 포트 사용 확인
!lsof -i :8000
```

### 런타임 연결 끊김
- 코드 셀에 주기적으로 실행 (Colab 자동 연결 해제 방지)
- 또는 브라우저 탭을 활성 상태로 유지